In [ ]:
from kalman import wrap_angle
from local_navigation import prox
import numpy as np

v_left = 200
v_right = 200
state = 0       #0=goal tracking, 1=obstacle avoidance
pos_x = 0       #from vision
pos_y = 0       #from vision
angle = 0       #from vision
path = [(0,0), (50,50), (100,100)]  # path to follow  -- maybe pos=path[0]
goal_x = 100    #from global navigation
goal_y = 100    #from global navigation


obstThrL = 10      # low obstacle threshold to switch state 1->0
obstThrH = 20      # high obstacle threshold to switch state 0->1
Gaind = 0.5         # gain for angle goal tracking
Gainp = 0.5        # gain for distance to goal

def tracking(pos_x, pos_y, angle, heading_x, heading_y):
    # compute the angle to the goal
    delta_x = heading_x - pos_x
    delta_y = heading_y - pos_y
    drho = np.sqrt(delta_x**2 + delta_y**2)
    dtheta = np.arctan2(delta_y, delta_x)

    angle_diff = wrap_angle((dtheta - angle))
    
    return angle_diff, drho 

def angle_to_speed(angle_diff, Gain):
    # compute the speed commands based on the angle difference
    left_cmd = v_left - Gain * angle_diff
    right_cmd = v_right + Gain * angle_diff
    
    return left_cmd, right_cmd

def distance_to_speed(drho, Gainp):
    # compute the speed commands based on the distance to the goal
    left_cmd = v_left + Gainp * drho
    right_cmd = v_right + Gainp * drho
    
    return left_cmd, right_cmd

goal_x, goal_y = path[-1]  # last waypoint is the goal
while ((pos_x, pos_y) != (goal_x, goal_y)):
    i = 1 
    heading_x, heading_y = path[i]
    while ((pos_x, pos_y) != (heading_x, heading_y)):
        # acquisition from the proximity sensors to detect obstacles
        obst = [prox_horizontal[0], prox_horizontal[1], prox_horizontal[2], prox_horizontal[3], prox_horizontal[4]]
            
        if state == 0: 
            # switch from goal tracking to obst avoidance if obstacle detected
            if min(obst) > obstThrH:
                state = 1
        elif state == 1:
            if max(obst) < obstThrL:
                state = 0
                
        if  state == 0 :
            # goal tracking: turn toward the goal
            angle_diff, drho = tracking(pos_x, pos_y, angle, heading_x, heading_y)
        
        else:
            # obstacle avoidance
            left_cmd, right_cmd = prox(v_left, v_right, prox_horizontal)
    i += 1
    # get the next waypoint from the path
